In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
# Création du dossier 'interim' s'il n'existe pas pour stocker les fichiers téléchargés
git_folder = "Patricia-Promise-Immo"
folder_entries = "data"
base_dir = Path().resolve().parents[1]
data_dir = base_dir/git_folder/folder_entries
interim_dir = data_dir/"interim"
if not interim_dir.exists():
    interim_dir.mkdir(parents=True)

In [8]:
# Définition des chemins
base_dir = Path().resolve().parents[1]
print(f"Base directory: {base_dir}")
data_dir = base_dir /"Patricia-Promise-Immo"/"data" /"raw_csv"
print(f"Interim directory: {data_dir}")
data_path = data_dir / "immo_entries_2025.csv"
insee_path = data_dir/"communes_france_2025.csv"

Base directory: D:\ProjectFolderDevAI_2025-2026\Immo_project
Interim directory: D:\ProjectFolderDevAI_2025-2026\Immo_project\Patricia-Promise-Immo\data\raw_csv


In [6]:
df = pd.read_csv(f"{data_path}", low_memory=False, sep=",")
df

,Identifiant de document,Reference document,1 Articles CGI,2 Articles CGI,3 Articles CGI,4 Articles CGI,5 Articles CGI,No disposition,Date mutation,Nature mutation,...,Surface Carrez du 5eme lot,Nombre de lots,Code type local,Type local,Identifiant local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,J,NaN,78.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,1.0,Maison,NaN,111.0,5.0,S,NaN,133.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,3.0,Dépendance,NaN,0.0,0.0,S,NaN,133.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,S,NaN,46.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,J,NaN,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,27/06/2025,Vente,...,NaN,2,2.0,Appartement,NaN,61.0,3.0,NaN,NaN,NaN
1387073,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,27/06/2025,Vente,...,NaN,2,2.0,Appartement,NaN,47.0,2.0,NaN,NaN,NaN
1387074,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,27/06/2025,Vente,...,NaN,2,3.0,Dépendance,NaN,0.0,0.0,NaN,NaN,NaN
1387075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,25/06/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# chargement communes et selection des colonnes uties
df2 = pd.read_csv(f"{insee_path}", on_bad_lines='skip', low_memory=False, sep = ",")
df2 = df2[['code_insee','population','superficie_hectare','densite','altitude_moyenne','altitude_minimale','altitude_maximale','latitude_mairie','longitude_mairie','latitude_centre','longitude_centre']]
df2

,code_insee,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
0,01001,832,1565,53.0,242,206.0,272.0,46.151,4.921,46.153,4.926
1,01002,267,912,29.0,483,290.0,748.0,46.007,5.423,46.009,5.428
2,01004,14854,2448,607.0,379,237.0,753.0,45.958,5.360,45.961,5.373
3,01005,1897,1605,118.0,290,265.0,302.0,45.996,4.903,45.996,4.912
4,01006,113,602,19.0,589,330.0,940.0,45.748,5.601,45.750,5.594
...,...,...,...,...,...,...,...,...,...,...,...
34930,97613,6432,2155,298.0,96,0.0,290.0,-12.761,45.084,-12.751,45.087
34931,97614,10203,1828,558.0,175,5.0,580.0,-12.849,45.139,-12.837,45.138
34932,97615,11442,426,2686.0,52,0.0,202.0,-12.798,45.275,-12.796,45.284
34933,97616,11156,1085,1028.0,130,0.0,510.0,-12.847,45.106,-12.861,45.119


In [ ]:
""" Fusion des datasets de vente et des communes"""
# Création du code INSEE
df['Code INSEE'] = (
    df['Code departement'].astype(str).str.zfill(2) + 
    df['Code commune'].astype(str).str.zfill(3)
)
# chargement communes
df2 = pd.read_csv(f"{insee_path}", on_bad_lines='skip', low_memory=False, sep = ",", engine="python")
df2 = df2[['code_insee','population','superficie_hectare','densite','altitude_moyenne','altitude_minimale','altitude_maximale','latitude_mairie','longitude_mairie','latitude_centre','longitude_centre']]

# merge
df_merged = pd.merge(
    df, df2,
    left_on='Code INSEE',
    right_on='code_insee',
    how='left'   # ou 'inner' selon ton besoin
)
data_interim_path = interim_dir/"all_immo_entries.csv"
df_merged.to_csv(data_interim_path, index=False)

In [17]:
data_interim_path = interim_dir/"all_immo_entries.csv"
# lecture du dataframe
df = pd.read_csv(f"{data_interim_path}", nrows=1_000_000, on_bad_lines='skip', low_memory=False, sep=',', dtype={'Code postal':str, 'Valeur fonciere':np.float64})

emptyness_infos = []
# nombre total de lignes
total_rows = df.shape[0] 
# liste des colonnes
columns = df.columns.tolist()

# pour chaque colones calcul du pourcentage de vide
for column in columns: 
    column_emptyness = df[df[column].isna()].shape[0] #nombre total de lignes vide dans la colone
    column_emptyness_rate = column_emptyness / total_rows * 100

    info = {
        'name': column,
        'pourcentage_vide' : column_emptyness_rate
    }

    emptyness_infos.append(info)
    

In [16]:
df_emptyness =  pd.DataFrame(emptyness_infos)

df_emptyness

,name,pourcentage_vide
0,Identifiant de document,100.0000
1,Reference document,100.0000
2,1 Articles CGI,100.0000
3,2 Articles CGI,100.0000
4,3 Articles CGI,100.0000
5,4 Articles CGI,100.0000
6,5 Articles CGI,100.0000
7,No disposition,0.0000
8,Date mutation,0.0000
9,Nature mutation,0.0000


In [15]:
num_cols_to_clean = [c for c in ["Valeur fonciere", "Surface reelle bati"] if c in df.columns]
for c in num_cols_to_clean:
    df[c] = (df[c].astype(str)
                 .str.replace(r"[^\d,.\-]", "", regex=True)
                 .str.replace(",", ".", regex=False))   # virgule -> point
    df[c] = pd.to_numeric(df[c], errors="coerce")
print(df.head())

   Identifiant de document  Reference document  1 Articles CGI  \
0                      NaN                 NaN             NaN   
1                      NaN                 NaN             NaN   
2                      NaN                 NaN             NaN   
3                      NaN                 NaN             NaN   
4                      NaN                 NaN             NaN   

   2 Articles CGI  3 Articles CGI  4 Articles CGI  5 Articles CGI  \
0             NaN             NaN             NaN             NaN   
1             NaN             NaN             NaN             NaN   
2             NaN             NaN             NaN             NaN   
3             NaN             NaN             NaN             NaN   
4             NaN             NaN             NaN             NaN   

   No disposition Date mutation Nature mutation  ...  population  \
0               1    02/01/2024           Vente  ...       124.0   
1               2    03/01/2024           Vente  ...

In [38]:
def get_year_from_date(df: pd.DataFrame)->pd.DataFrame:
    cols = {c.lower().strip(): c for c in df.columns}
    print(cols)
    if "date mutation" not in cols:
        print("La colonne \"Date mutation\" n'existe pas dans le DataFrame")
    date_col = cols["date mutation"]
    s = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)
    print(s)
    df["years"] = s.dt.year.astype("Int16")
    return df

In [39]:
get_year_from_date(df)

{'identifiant de document': 'Identifiant de document', 'reference document': 'Reference document', '1 articles cgi': '1 Articles CGI', '2 articles cgi': '2 Articles CGI', '3 articles cgi': '3 Articles CGI', '4 articles cgi': '4 Articles CGI', '5 articles cgi': '5 Articles CGI', 'no disposition': 'No disposition', 'date mutation': 'Date mutation', 'nature mutation': 'Nature mutation', 'valeur fonciere': 'Valeur fonciere', 'no voie': 'No voie', 'b/t/q': 'B/T/Q', 'type de voie': 'Type de voie', 'code voie': 'Code voie', 'voie': 'Voie', 'code postal': 'Code postal', 'commune': 'Commune', 'code departement': 'Code departement', 'code commune': 'Code commune', 'prefixe de section': 'Prefixe de section', 'section': 'Section', 'no plan': 'No plan', 'no volume': 'No Volume', '1er lot': '1er lot', 'surface carrez du 1er lot': 'Surface Carrez du 1er lot', '2eme lot': '2eme lot', 'surface carrez du 2eme lot': 'Surface Carrez du 2eme lot', '3eme lot': '3eme lot', 'surface carrez du 3eme lot': 'Surf

,Identifiant de document,Reference document,1 Articles CGI,2 Articles CGI,3 Articles CGI,4 Articles CGI,5 Articles CGI,No disposition,Date mutation,Nature mutation,...,Nombre de lots,Code type local,Type local,Identifiant local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain,years
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,0,NaN,NaN,NaN,NaN,NaN,J,NaN,78.0,2025
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,0,1.0,Maison,NaN,111.0,5.0,S,NaN,133.0,2025
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,0,3.0,Dépendance,NaN,0.0,0.0,S,NaN,133.0,2025
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,0,NaN,NaN,NaN,NaN,NaN,S,NaN,46.0,2025
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,0,NaN,NaN,NaN,NaN,NaN,J,NaN,17.0,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,27/06/2025,Vente,...,2,2.0,Appartement,NaN,61.0,3.0,NaN,NaN,NaN,2025
1387073,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,27/06/2025,Vente,...,2,2.0,Appartement,NaN,47.0,2.0,NaN,NaN,NaN,2025
1387074,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,27/06/2025,Vente,...,2,3.0,Dépendance,NaN,0.0,0.0,NaN,NaN,NaN,2025
1387075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,25/06/2025,Vente,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025


In [11]:
# --- Taux de vide simple
emptyness = df.isna().mean().mul(100).rename("pourcentage_vide").reset_index()
emptyness.columns = ["name", "pourcentage_vide"]

# Colonnes à garder (<100% vide)
assez_rempli = emptyness.loc[emptyness["pourcentage_vide"] < 100, "name"].tolist()


In [12]:
assez_rempli

['No disposition',
 'Date mutation',
 'Nature mutation',
 'Valeur fonciere',
 'No voie',
 'B/T/Q',
 'Type de voie',
 'Code voie',
 'Voie',
 'Code postal',
 'Commune',
 'Code departement',
 'Code commune',
 'Prefixe de section',
 'Section',
 'No plan',
 'No Volume',
 '1er lot',
 'Surface Carrez du 1er lot',
 '2eme lot',
 'Surface Carrez du 2eme lot',
 '3eme lot',
 'Surface Carrez du 3eme lot',
 '4eme lot',
 'Surface Carrez du 4eme lot',
 '5eme lot',
 'Surface Carrez du 5eme lot',
 'Nombre de lots',
 'Code type local',
 'Type local',
 'Surface reelle bati',
 'Nombre pieces principales',
 'Nature culture',
 'Nature culture speciale',
 'Surface terrain',
 'Code INSEE',
 'code_insee',
 'population',
 'superficie_hectare',
 'densite',
 'altitude_moyenne',
 'altitude_minimale',
 'altitude_maximale',
 'latitude_mairie',
 'longitude_mairie',
 'latitude_centre',
 'longitude_centre']

In [14]:
rempli

NameError: name 'rempli' is not defined

In [18]:
completement_vide

NameError: name 'completement_vide' is not defined

In [19]:
print(len(assez_rempli))
assez_rempli

47


['No disposition',
 'Date mutation',
 'Nature mutation',
 'Valeur fonciere',
 'No voie',
 'B/T/Q',
 'Type de voie',
 'Code voie',
 'Voie',
 'Code postal',
 'Commune',
 'Code departement',
 'Code commune',
 'Prefixe de section',
 'Section',
 'No plan',
 'No Volume',
 '1er lot',
 'Surface Carrez du 1er lot',
 '2eme lot',
 'Surface Carrez du 2eme lot',
 '3eme lot',
 'Surface Carrez du 3eme lot',
 '4eme lot',
 'Surface Carrez du 4eme lot',
 '5eme lot',
 'Surface Carrez du 5eme lot',
 'Nombre de lots',
 'Code type local',
 'Type local',
 'Surface reelle bati',
 'Nombre pieces principales',
 'Nature culture',
 'Nature culture speciale',
 'Surface terrain',
 'Code INSEE',
 'code_insee',
 'population',
 'superficie_hectare',
 'densite',
 'altitude_moyenne',
 'altitude_minimale',
 'altitude_maximale',
 'latitude_mairie',
 'longitude_mairie',
 'latitude_centre',
 'longitude_centre']

In [20]:
data_no_empty_path = interim_dir / "all_immo_no_empty.csv"
df[assez_rempli].to_csv(data_no_empty_path, index=False)
print(f"Saved: {data_no_empty_path}")


Saved: D:\ProjectFolderDevAI_2025-2026\Immo_project\Patricia-Promise-Immo\data\interim\all_immo_no_empty.csv


In [21]:
df = pd.read_csv(f"{data_no_empty_path}",
    sep=",",              # séparateur
    quotechar='"',        # gère les guillemets
    na_filter=True,       # détecte les valeurs manquantes
    low_memory=False,
    dtype={"Code postal": str} 
)
df

,No disposition,Date mutation,Nature mutation,Valeur fonciere,No voie,B/T/Q,Type de voie,Code voie,Voie,Code postal,...,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
0,1,02/01/2024,Vente,346.5,NaN,NaN,NaN,B020,LE DELIVRE,1230,...,124.0,465.0,27.0,591.0,394.0,910.0,45.955,5.530,45.953,5.537
1,2,03/01/2024,Vente,10000.0,NaN,NaN,NaN,B007,CHEVRY DESSOUS,1170,...,2261.0,579.0,391.0,496.0,457.0,580.0,46.280,6.036,46.283,6.047
2,1,08/01/2024,Vente,249000.0,NaN,NaN,NaN,B086,PIN HAMEAU,1290,...,1299.0,1029.0,126.0,186.0,173.0,210.0,46.248,4.890,46.247,4.899
3,1,03/01/2024,Vente,329500.0,29.0,NaN,PL,0500,DU JURA,1170,...,13078.0,3186.0,410.0,949.0,532.0,1614.0,46.334,6.058,46.347,6.046
4,1,03/01/2024,Vente,329500.0,9001.0,NaN,PL,0500,DU JURA,1170,...,13078.0,3186.0,410.0,949.0,532.0,1614.0,46.334,6.058,46.347,6.046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,1,22/03/2024,Vente,131850.0,6.0,B,AV,0067,DE L'AERODROME DE MONTAUDR,31400,...,504078.0,11809.0,4269.0,148.0,115.0,263.0,43.605,1.444,43.596,1.432
999996,1,22/03/2024,Vente,131850.0,6.0,NaN,AV,0067,DE L'AERODROME DE MONTAUDR,31400,...,504078.0,11809.0,4269.0,148.0,115.0,263.0,43.605,1.444,43.596,1.432
999997,1,13/03/2024,Vente,300000.0,42.0,NaN,RUE,2424,DE CUGNAUX,31300,...,504078.0,11809.0,4269.0,148.0,115.0,263.0,43.605,1.444,43.596,1.432
999998,1,13/03/2024,Vente,300000.0,42.0,NaN,RUE,2424,DE CUGNAUX,31300,...,504078.0,11809.0,4269.0,148.0,115.0,263.0,43.605,1.444,43.596,1.432


In [22]:
if df.columns[0].startswith("Unnamed"):
    df = df.drop(columns=df.columns[0])
df

,No disposition,Date mutation,Nature mutation,Valeur fonciere,No voie,B/T/Q,Type de voie,Code voie,Voie,Code postal,...,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
0,1,02/01/2024,Vente,346.5,NaN,NaN,NaN,B020,LE DELIVRE,1230,...,124.0,465.0,27.0,591.0,394.0,910.0,45.955,5.530,45.953,5.537
1,2,03/01/2024,Vente,10000.0,NaN,NaN,NaN,B007,CHEVRY DESSOUS,1170,...,2261.0,579.0,391.0,496.0,457.0,580.0,46.280,6.036,46.283,6.047
2,1,08/01/2024,Vente,249000.0,NaN,NaN,NaN,B086,PIN HAMEAU,1290,...,1299.0,1029.0,126.0,186.0,173.0,210.0,46.248,4.890,46.247,4.899
3,1,03/01/2024,Vente,329500.0,29.0,NaN,PL,0500,DU JURA,1170,...,13078.0,3186.0,410.0,949.0,532.0,1614.0,46.334,6.058,46.347,6.046
4,1,03/01/2024,Vente,329500.0,9001.0,NaN,PL,0500,DU JURA,1170,...,13078.0,3186.0,410.0,949.0,532.0,1614.0,46.334,6.058,46.347,6.046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,1,22/03/2024,Vente,131850.0,6.0,B,AV,0067,DE L'AERODROME DE MONTAUDR,31400,...,504078.0,11809.0,4269.0,148.0,115.0,263.0,43.605,1.444,43.596,1.432
999996,1,22/03/2024,Vente,131850.0,6.0,NaN,AV,0067,DE L'AERODROME DE MONTAUDR,31400,...,504078.0,11809.0,4269.0,148.0,115.0,263.0,43.605,1.444,43.596,1.432
999997,1,13/03/2024,Vente,300000.0,42.0,NaN,RUE,2424,DE CUGNAUX,31300,...,504078.0,11809.0,4269.0,148.0,115.0,263.0,43.605,1.444,43.596,1.432
999998,1,13/03/2024,Vente,300000.0,42.0,NaN,RUE,2424,DE CUGNAUX,31300,...,504078.0,11809.0,4269.0,148.0,115.0,263.0,43.605,1.444,43.596,1.432
